# Bengali Hallucination Detection — Offline Kaggle GPU Pipeline

Self-contained offline inference notebook for **অলীকবচন** (IUT ICT Fest Datathon 2026).  
Metric target: **F1 on hallucinated class (`label=0`)**. Models must be open-weight and loaded from attached Kaggle datasets (no external APIs).

This variant adds a **soft Gemma-4 judge** (first-token logprobs for FAITHFUL vs HALLUCINATED) on top of E5 retrieval + mDeBERTa NLI. Attach a local `google/gemma-4-E4B-it` model dataset and update `GEMMA_MODEL_PATH` before running.

## Method (this notebook)

1. **Character TF-IDF** over question + answer + context (surface / template signal).
2. **Open-book path:** E5 ranks context sentences; mDeBERTa-XNLI scores entailment/contradiction on top-k evidence (answer-only and Q+A hypotheses).
3. **Closed/open dual thresholds:** class-0 F1 threshold selected separately for rows with and without context.
4. **Labeled answer memory:** exact train prompt/answer matches are reused deterministically; soft similarities are features only.
5. **Gemma-4 soft judge:** 4-bit offline model produces continuous P(hallucinated), hard vote, and logit margin.

## Before running

1. Set **Accelerator** to **GPU T4 x2** or **GPU P100**.
2. Attach the competition dataset plus local datasets for E5 and mDeBERTa NLI **and Gemma-4-E4B-it**.
3. Edit model paths in the configuration cell only if your attached dataset paths differ.
4. If package install needs Internet, run once online, then re-run offline with Internet disabled for the final submission version.
5. **Run All**. Final cell writes `submission.csv`.

Runtime budget: under 9 hours. Combined model disk size must stay under 50 GB.


In [ ]:
import os
import sys
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    if torch.cuda.device_count() > 1:
        print("GPU count:", torch.cuda.device_count())
else:
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")


## Install packages

Kaggle images often already include these. If install needs Internet, enable it once, save the version, then for the official offline re-run attach wheels as a Dataset and use `--no-index --find-links`.


In [ ]:
%pip install -q "transformers>=5.10.1" accelerate bitsandbytes scikit-learn scipy pandas matplotlib seaborn

## Configure attached models and locate competition files

Only edit model path variables if your Kaggle dataset slugs differ. Paths must point to **model directories**, not single files. Competition `dataset_samples.json` / `test.csv` are discovered automatically.


In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
# Update these if your attached dataset paths differ:
EMBED_MODEL_PATH = "/kaggle/input/models/petrcher/multilingual-e5-large-instruct/transformers/default/1"
NLI_MODEL_PATH = "/kaggle/input/datasets/nhenseler/mdebertav3basemnlixnli"
BATCH_SIZE = 12

def find_input_file(names):
    roots = [Path.cwd(), KAGGLE_INPUT, Path("/content")]
    matches = []
    for root in roots:
        if root.exists():
            for name in names:
                matches.extend(root.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find any of: {names}")
    return str(sorted(matches)[0])

TRAIN_PATH = find_input_file(["dataset_samples.json", "dataset samples.json"])
TEST_PATH = find_input_file(["test.csv", "test set.csv"])

for path in [TRAIN_PATH, TEST_PATH, EMBED_MODEL_PATH, NLI_MODEL_PATH]:
    print(path, "exists:", Path(path).exists())

assert Path(EMBED_MODEL_PATH).is_dir(), "Set EMBED_MODEL_PATH to your attached E5 model folder."
assert Path(NLI_MODEL_PATH).is_dir(), "Set NLI_MODEL_PATH to your attached NLI model folder."

GEMMA_MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
GEMMA_BATCH_SIZE = 4
assert Path(GEMMA_MODEL_PATH).is_dir(), "Attach google/gemma-4-E4B-it and set GEMMA_MODEL_PATH."


## Pipeline imports, text normalizations

Basic text sanitization, NFKC normalization, and tokenizing keys.


In [ ]:
"""Offline Kaggle inference pipeline for Bengali hallucination detection.

Attach model folders to the Kaggle notebook, disable internet, then run:
python high_score_pipeline.py --train /kaggle/input/.../dataset_samples.json \
  --test /kaggle/input/.../test.csv --output submission.csv \
  --embed-model /kaggle/input/multilingual-e5-large-instruct \
  --nli-model /kaggle/input/mdeberta-v3-xnli \
  [--gemma-model /kaggle/input/gemma-4-e4b-it]

Targets official class-0 (hallucinated) F1. Open-weight / offline only.
"""

from __future__ import annotations

import argparse
import json
import re
import unicodedata
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)



NULL_CONTEXTS = {"", "[NULL]", "nan", "none", "null"}
EVIDENCE_TOP_K = 4
GEMMA_FEATURE_DIM = 3

# ---------------------------------------------------------------------------
# EXPERIMENT TOGGLES. The defaults below reproduce the 0.722 baseline exactly.
# Change ONE knob per submission: on ~300 training rows the OOF score does not
# track the leaderboard, so each idea must be judged by an actual submission.
# ---------------------------------------------------------------------------
# (1) Per-claim NLI: split the answer into claims and keep the worst-supported
#     one (min entailment / max contradiction). Catches a single false fact that
#     whole-answer NLI dilutes. Adds a zero-shot signal, not trained capacity.
PER_CLAIM_NLI = True
MAX_ANSWER_CLAIMS = 4          # runtime guard on the extra NLI pairs
# (2) Regularization of the tiny-data TF-IDF + LR head (lower = less memorizing).
LR_C = 1.0
TFIDF_MAX_FEATURES = 4000      # capacity cut: 120k char n-grams on ~300 rows memorizes
# (3) Post-selection threshold nudges to probe decision-boundary sensitivity.
THR_OPEN_DELTA = 0.0
THR_CLOSED_DELTA = 0.0
# (4) Threshold selection method. single = one-shot OOF argmax; nested = per-fold avg.
THRESHOLD_METHOD = "nested"
# (5) Retrieval and self-consistency parameters
HYBRID_RETRIEVAL = True
GEMMA_NUM_SAMPLES = 1
# (6) Route Gemma over retrieved top-k evidence instead of raw truncated context.
GEMMA_USE_RETRIEVED_EVIDENCE = True
# (7) Fuzzy near-match override. Enable only if override diagnostic shows >= 10 percent test rows.
NEAR_MATCH_OVERRIDE = False
NEAR_MATCH_SIM_THRESHOLD = 0.90
NEAR_MATCH_LABEL_CONFIDENCE = 0.70

# (4) Retrieval and self-consistency parameters

# (8) Causal-LM judge (replaces old multimodal GemmaJudge path).
JUDGE_KIND = "aya8b"      # aya8b | qwen14b | any HF causal LM
JUDGE_MODEL_PATH = "/kaggle/input/models/cohereai/aya-expanse-8b/transformers/default/1"
JUDGE_BATCH_SIZE = 4      # halve to 2 on 14B
JUDGE_MAX_CONTEXT_CHARS = 1600
BASE_NLI_FEATURE_DIM = 12
PER_CLAIM_FEATURE_DIM = 4
NLI_FEATURE_DIM = BASE_NLI_FEATURE_DIM + (PER_CLAIM_FEATURE_DIM if PER_CLAIM_NLI else 0)


def normalize_text(value: object) -> str:
    text = unicodedata.normalize("NFKC", str(value if value is not None else ""))
    text = text.replace("ড়", "ড়").replace("ঢ়", "ঢ়")
    return re.sub(r"\s+", " ", text).strip()


def key_text(value: object) -> str:
    text = normalize_text(value).lower()
    return re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE).strip()


def has_context(value: object) -> bool:
    return key_text(value) not in NULL_CONTEXTS



## Data Loading Utility

Loads train (JSON) and test (CSV) sets, normalizing all text fields.


In [ ]:
def load_data(train_path: str, test_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    with open(train_path, encoding="utf-8") as file:
        train = pd.DataFrame(json.load(file))
    test = pd.read_csv(test_path)
    for frame in (train, test):
        for column in ("context", "prompt_bn", "response_bn"):
            frame[column] = frame[column].fillna("").map(normalize_text)
        frame["has_context"] = frame["context"].map(has_context)
        frame["prompt_key"] = frame["prompt_bn"].map(key_text)
        frame["answer_key"] = frame["response_bn"].map(key_text)
    train["label"] = train["label"].astype(int)
    return train, test


def make_text(frame: pd.DataFrame) -> list[str]:
    contexts = frame["context"].where(frame["has_context"], "[NO_CONTEXT]")
    return (
        "[QUESTION] " + frame["prompt_bn"] + " [ANSWER] " + frame["response_bn"]
        + " [CONTEXT] " + contexts.str.slice(0, 2400)
    ).tolist()



## Text Similarity & Reference Answer Match

Uses SequenceMatcher and token Jaccard similarity. Generates features checking if a prompt was seen in the training data, allowing exact reuse or similarity matching.


In [ ]:
def token_jaccard(first: str, second: str) -> float:
    first_tokens = set(first.split())
    second_tokens = set(second.split())
    if not first_tokens or not second_tokens:
        return 0.0
    return len(first_tokens & second_tokens) / len(first_tokens | second_tokens)


def text_similarity(first: str, second: str) -> float:
    if not first or not second:
        return 0.0
    if first == second:
        return 1.0
    seq = SequenceMatcher(None, first, second).ratio()
    return max(token_jaccard(first, second), float(seq))



def answer_reference_features(
    train: pd.DataFrame, frame: pd.DataFrame, exclude_same_index: bool = False
) -> np.ndarray:
    """Nearest labeled answers for the same prompt (valid supervised retrieval)."""
    references: dict[str, list[tuple[int, str, int]]] = {}
    for row in train.itertuples():
        references.setdefault(row.prompt_key, []).append((row.Index, row.answer_key, row.label))
    values = np.zeros((len(frame), 7), dtype=np.float32)
    for index, row in enumerate(frame.itertuples()):
        candidates = references.get(row.prompt_key, [])
        if exclude_same_index:
            candidates = [candidate for candidate in candidates if candidate[0] != row.Index]
        if not candidates:
            continue
        similarities = [
            (text_similarity(row.answer_key, answer), label, row.answer_key == answer)
            for _, answer, label in candidates
        ]
        positive = [score for score, label, _ in similarities if label == 1]
        negative = [score for score, label, _ in similarities if label == 0]
        best_score, best_label, _ = max(similarities, key=lambda item: item[0])
        values[index] = [
            1.0,
            max(positive, default=0.0),
            max(negative, default=0.0),
            float(any(exact and label == 1 for _, label, exact in similarities)),
            float(any(exact and label == 0 for _, label, exact in similarities)),
            best_score,
            float(best_label == 0),
        ]
    return values



## Sentence splitting & Empty Cache utilities

Splits context passages into sentences, merging short segments to provide usable premises for NLI.


In [ ]:
def split_sentences(context: str) -> list[str]:
    pieces = [piece.strip() for piece in re.split(r"(?<=[।!?\n])\s+", context) if piece.strip()]
    if not pieces:
        pieces = [context.strip()] if context.strip() else []
    # Merge very short fragments so NLI gets usable premises.
    merged: list[str] = []
    buffer = ""
    for piece in pieces:
        if len(piece) < 40:
            buffer = f"{buffer} {piece}".strip()
            if len(buffer) >= 40:
                merged.append(buffer)
                buffer = ""
        else:
            if buffer:
                merged.append(buffer)
                buffer = ""
            merged.append(piece)
    if buffer:
        merged.append(buffer)
    return merged or ([context[:1800]] if context else [])


def empty_cache() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class MiniBM25:
    """Lightweight native BM25 implementation for ranking short sentences."""
    def __init__(self, corpus: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_len = [len(doc.split()) for doc in corpus]
        self.avg_doc_len = sum(self.doc_len) / max(1, self.corpus_size)
        self.doc_freqs = []
        self.idf = {}
        
        df = {}
        for doc in corpus:
            freqs = {}
            for word in doc.split():
                freqs[word] = freqs.get(word, 0) + 1
            self.doc_freqs.append(freqs)
            for word in freqs:
                df[word] = df.get(word, 0) + 1
                
        for word, count in df.items():
            self.idf[word] = float(np.log((self.corpus_size - count + 0.5) / (count + 0.5) + 1.0))

    def score(self, query: str) -> np.ndarray:
        query_words = query.split()
        scores = np.zeros(self.corpus_size, dtype=np.float32)
        for i in range(self.corpus_size):
            freqs = self.doc_freqs[i]
            d_len = self.doc_len[i]
            score = 0.0
            for word in query_words:
                if word in freqs:
                    tf = freqs[word]
                    idf = self.idf.get(word, 0.0)
                    denom = tf + self.k1 * (1.0 - self.b + self.b * (d_len / self.avg_doc_len))
                    score += idf * (tf * (self.k1 + 1.0)) / denom
            scores[i] = score
        return scores



## Exploratory Data Analysis (EDA)

Visualize target label balance in the training set and compare the count of open-book vs. closed-book rows in train and test datasets.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load data
train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)

# Set style
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Label distribution
sns.countplot(x="label", data=train_df, ax=axes[0], hue="label", legend=False)
axes[0].set_title("Training Set Label Distribution\n(0 = Hallucinated, 1 = Faithful)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="center", xytext=(0, 5), textcoords="offset points")

# 2. Context existence
train_context = pd.DataFrame({"source": "Train", "has_context": train_df["has_context"]})
test_context = pd.DataFrame({"source": "Test", "has_context": test_df["has_context"]})
combined_context = pd.concat([train_context, test_context])

sns.countplot(x="source", hue="has_context", data=combined_context, ax=axes[1])
axes[1].set_title("Context Existence Comparison (Train vs Test)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Dataset Split")
axes[1].set_ylabel("Count")
axes[1].legend(title="Has Context", labels=["No (Closed-book)", "Yes (Open-book)"])
for p in axes[1].patches:
    if p.get_height() > 0:
        axes[1].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="center", xytext=(0, 5), textcoords="offset points")

plt.tight_layout()
plt.savefig("dataset_eda.png", dpi=150, bbox_inches="tight")
plt.show()


## Evidence retrieval: E5 Sentence Retriever

Uses multilingual E5 large to retrieve and rank context sentences based on prompt and response query.


In [ ]:
@dataclass
class E5Embedder:
    model_path: str
    device: str
    batch_size: int

    def __post_init__(self) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, local_files_only=True)
        self.model = AutoModel.from_pretrained(self.model_path, local_files_only=True).to(self.device).eval()

    def encode(self, texts: list[str]) -> np.ndarray:
        if not texts:
            return np.zeros((0, self.model.config.hidden_size), dtype=np.float32)
        embeddings = []
        with torch.inference_mode():
            for start in range(0, len(texts), self.batch_size):
                inputs = self.tokenizer(
                    texts[start:start + self.batch_size], padding=True, truncation=True,
                    max_length=512, return_tensors="pt"
                ).to(self.device)
                token_embeddings = self.model(**inputs).last_hidden_state
                mask = inputs["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
                pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                embeddings.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
        return np.concatenate(embeddings, axis=0)



## Verification: Multilingual NLI Verifier

Uses mDeBERTa-v3 to compute entailment and contradiction scores between context sentences (premises) and prompt/response (hypotheses).


In [ ]:
@dataclass
class Verifier:
    """Open-book E5 sentence retrieval + multilingual NLI support features."""

    embed_model_path: str
    nli_model_path: str
    batch_size: int = 12
    top_k: int = EVIDENCE_TOP_K

    def __post_init__(self) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embedder = E5Embedder(self.embed_model_path, self.device, self.batch_size)
        self.tokenizer = AutoTokenizer.from_pretrained(self.nli_model_path, local_files_only=True)
        self.nli = AutoModelForSequenceClassification.from_pretrained(
            self.nli_model_path, local_files_only=True
        ).to(self.device).eval()
        labels = {str(name).lower(): int(index) for name, index in self.nli.config.label2id.items()}
        self.entailment_index = next(index for name, index in labels.items() if "entail" in name)
        self.contradiction_index = next(index for name, index in labels.items() if "contrad" in name)

    def _nli_batch(self, premises: list[str], hypotheses: list[str]) -> np.ndarray:
        if not premises:
            return np.zeros((0, self.nli.config.num_labels), dtype=np.float32)
        scores = []
        with torch.inference_mode():
            for start in range(0, len(premises), self.batch_size):
                inputs = self.tokenizer(
                    premises[start:start + self.batch_size],
                    hypotheses[start:start + self.batch_size],
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(self.device)
                scores.append(torch.softmax(self.nli(**inputs).logits, dim=-1).cpu().numpy())
        return np.concatenate(scores, axis=0)

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        output = np.zeros((len(frame), NLI_FEATURE_DIM), dtype=np.float32)
        contextual_rows = np.flatnonzero(frame["has_context"].to_numpy())
        if not len(contextual_rows):
            return output

        sentences_by_row: dict[int, list[str]] = {}
        all_passages: list[str] = []
        passage_owner: list[int] = []
        for index in contextual_rows:
            sentences = split_sentences(frame.iloc[index].context)
            # Cap very long contexts to keep runtime inside Kaggle limits.
            if len(sentences) > 40:
                sentences = sentences[:40]
            sentences_by_row[index] = sentences
            for sentence in sentences:
                all_passages.append(f"passage: {sentence}")
                passage_owner.append(index)

        query_text = [
            (
                "Instruct: Retrieve evidence that verifies a Bengali question and answer.\n"
                f"Query: {frame.iloc[index].prompt_bn} উত্তর: {frame.iloc[index].response_bn}"
            )
            for index in contextual_rows
        ]
        answer_text = [f"passage: {frame.iloc[index].response_bn}" for index in contextual_rows]
        queries = self.embedder.encode(query_text)
        answers = self.embedder.encode(answer_text)
        passage_embeddings = self.embedder.encode(all_passages)

        # Map each contextual row -> passage embedding slice
        offsets: dict[int, list[int]] = {}
        for pos, owner in enumerate(passage_owner):
            offsets.setdefault(owner, []).append(pos)

        nli_pairs: list[tuple[int, str, str]] = []
        claim_pairs: list[tuple[int, int, str, str]] = []  # (row, claim_id, premise, claim)
        evidence_sims: dict[int, np.ndarray] = {}
        top_evidence_text: dict[int, str] = {}
        for local_idx, index in enumerate(contextual_rows):
            positions = offsets[index]
            row_emb = passage_embeddings[positions]
            sims = row_emb @ queries[local_idx]
            
            if HYBRID_RETRIEVAL:
                # E5 ranks
                rank_e5 = np.argsort(-sims)
                rank_e5_pos = np.zeros(len(sims), dtype=int)
                for pos, idx_s in enumerate(rank_e5):
                    rank_e5_pos[idx_s] = pos
                    
                # BM25 ranks
                bm25 = MiniBM25([key_text(s) for s in sentences_by_row[index]])
                query = f"{frame.iloc[index].prompt_bn} {frame.iloc[index].response_bn}"
                bm25_scores = bm25.score(key_text(query))
                rank_bm25 = np.argsort(-bm25_scores)
                rank_bm25_pos = np.zeros(len(sims), dtype=int)
                for pos, idx_s in enumerate(rank_bm25):
                    rank_bm25_pos[idx_s] = pos
                    
                # RRF Fusion
                rrf_k = 60
                rrf_scores = np.zeros(len(sims), dtype=np.float32)
                for j in range(len(sims)):
                    rrf_scores[j] = 1.0 / (rrf_k + rank_e5_pos[j]) + 1.0 / (rrf_k + rank_bm25_pos[j])
                    
                order = np.argsort(-rrf_scores)[: min(self.top_k, len(sims))]
            else:
                order = np.argsort(-sims)[: min(self.top_k, len(sims))]
                
            evidence_sims[index] = np.asarray(sims[order], dtype=np.float32)
            selected = [sentences_by_row[index][int(i)] for i in order]
            top_evidence_text[index] = " ".join(selected)
            hypothesis_answer = frame.iloc[index].response_bn
            hypothesis_qa = (
                f"প্রশ্ন: {frame.iloc[index].prompt_bn} উত্তর: {frame.iloc[index].response_bn}"
            )
            for chunk in selected:
                nli_pairs.append((index, chunk, hypothesis_answer))
                nli_pairs.append((index, chunk, hypothesis_qa))
            if PER_CLAIM_NLI:
                claims = split_sentences(frame.iloc[index].response_bn)[:MAX_ANSWER_CLAIMS]
                claims = claims or [frame.iloc[index].response_bn]
                for claim_id, claim in enumerate(claims):
                    for chunk in selected:
                        claim_pairs.append((index, claim_id, chunk, claim))

        premises = [pair[1] for pair in nli_pairs]
        hypotheses = [pair[2] for pair in nli_pairs]
        probabilities = self._nli_batch(premises, hypotheses)

        by_row: dict[int, list[np.ndarray]] = {}
        for pair, probability in zip(nli_pairs, probabilities):
            by_row.setdefault(pair[0], []).append(probability)

        for local_idx, index in enumerate(contextual_rows):
            nli_scores = np.stack(by_row[index])
            entail = nli_scores[:, self.entailment_index]
            contrad = nli_scores[:, self.contradiction_index]
            sims = evidence_sims[index]
            answer_overlap = token_jaccard(
                key_text(frame.iloc[index].response_bn),
                key_text(top_evidence_text[index]),
            )
            output[index, :BASE_NLI_FEATURE_DIM] = [
                float(entail.max()),
                float(entail.mean()),
                float(entail.min()),
                float(contrad.max()),
                float(contrad.mean()),
                float(entail.max() - contrad.max()),
                float((contrad > 0.5).mean()),
                float((entail > 0.5).mean()),
                float(sims.max()) if len(sims) else 0.0,
                float(sims.mean()) if len(sims) else 0.0,
                float(queries[local_idx] @ answers[local_idx]),
                float(answer_overlap),
            ]

        if PER_CLAIM_NLI and claim_pairs:
            claim_probs = self._nli_batch(
                [pair[2] for pair in claim_pairs], [pair[3] for pair in claim_pairs]
            )
            # Best supporting evidence per claim, then aggregate across claims.
            per_claim: dict[tuple[int, int], list[np.ndarray]] = {}
            for pair, probability in zip(claim_pairs, claim_probs):
                per_claim.setdefault((pair[0], pair[1]), []).append(probability)
            per_row_claims: dict[int, list[tuple[float, float]]] = {}
            for (row_index, _), probs in per_claim.items():
                arr = np.stack(probs)
                best_entail = float(arr[:, self.entailment_index].max())
                worst_contrad = float(arr[:, self.contradiction_index].max())
                per_row_claims.setdefault(row_index, []).append((best_entail, worst_contrad))
            for row_index, claim_scores in per_row_claims.items():
                entails = np.asarray([score[0] for score in claim_scores], dtype=np.float32)
                contrads = np.asarray([score[1] for score in claim_scores], dtype=np.float32)
                output[row_index, BASE_NLI_FEATURE_DIM:] = [
                    float(entails.min()),        # weakest-supported claim
                    float(entails.mean()),       # average claim support
                    float(contrads.max()),       # most-contradicted claim
                    float((contrads > 0.5).mean()),  # fraction of claims contradicted
                ]
        # Stash the top-k retrieved evidence sentences so the caller can
        # route Gemma over them when GEMMA_USE_RETRIEVED_EVIDENCE is on.
        self.last_top_evidence = dict(top_evidence_text)

        return output

    def close(self) -> None:
        del self.nli
        del self.embedder.model
        del self.embedder
        empty_cache()



## Gemma-4 Soft Judge

Loads Gemma-4 in 4-bit quantization, formatting chat templates and retrieving soft hallucination probabilities using first-token logits.


In [ ]:
@dataclass
class CausalLMJudge:
    """Soft hallucination probability from a HuggingFace causal LM.
    First-token logits over FAITHFUL vs HALLUCINATED.

    Supports Aya-Expanse-8B, Qwen2.5-14B-Instruct, Gemma text, or any
    HuggingFace causal LM with a chat template. Offline-only, 4-bit (NF4).
    """

    model_path: str
    batch_size: int = 4
    max_context_chars: int = 1600
    num_samples: int = GEMMA_NUM_SAMPLES
    kind: str = "custom"

    def __post_init__(self) -> None:
        from transformers import (
            AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
        )
        if not torch.cuda.is_available():
            raise RuntimeError("Causal-LM judge requires a Kaggle GPU.")
        # bfloat16 compute is safer than fp16 across architectures (Aya,
        # Qwen, Gemma); fp16 occasionally NaNs on extreme logits.
        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path, local_files_only=True
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path, local_files_only=True, device_map="auto",
            quantization_config=quantization,
        ).eval()
        self.faithful_token_ids = self._label_token_ids("FAITHFUL")
        self.hallucinated_token_ids = self._label_token_ids("HALLUCINATED")

    def _label_token_ids(self, label: str) -> list[int]:
        # Some chat models encode the label as multiple subword pieces.
        # We need the FIRST token of the label, not the full encoding,
        # because we are scoring the very first generation step.
        ids = self.tokenizer.encode(label, add_special_tokens=False)
        alt = self.tokenizer.encode(" " + label, add_special_tokens=False)
        merged = []
        if ids: merged.append(ids[0])
        if alt: merged.append(alt[0])
        out = list(set(merged)) or [self.tokenizer.unk_token_id]
        vocab = self.tokenizer.vocab_size
        return [i for i in out if 0 <= i < vocab]

    def make_prompt(self, row: pd.Series) -> str:
        if row["has_context"]:
            retrieved = row.get("retrieved_evidence", None) if hasattr(row, "get") else None
            if isinstance(retrieved, str) and retrieved.strip():
                evidence = retrieved[: self.max_context_chars]
            else:
                evidence = row.context[: self.max_context_chars]
            task = (
                """ইরসইার:\n{evidence}\n\nইরশ্ম:\n{row.prompt_bn}\n\nএত্র:\n{row.response_bn}\n\n"""
                .format(evidence=evidence, row=row)
                + "নিয়ম: শুধু প্রসঙ্গ ব্যবহার করুন। প্রসঙ্গে সমর্থিত না হলে HALLUCINATED। উত্তর প্রশ্নের সাথে মিললে এবং প্রসঙ্গে থাকলে FAITHFUL।"
            )
        else:
            task = (
                """ইরশ্ম:\n{row.prompt_bn}\n\nএত্র:\n{row.response_bn}\n\n"""
                .format(row=row)
                + "নিয়ম: প্রতিষ্ঠিত সাধারণ জ্ঞান ও বাংলাদেশ/বাংলা সংস্কৃতির তথ্য ব্যবহার করুন। ভুল তথ্য, ভুল সংখ্যা, বা অসমর্থিত দাবি থাকলে HALLUCINATED, নইলে FAITHFUL।"
            )
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a precise Bengali hallucination detector. "
                    + "Reply with exactly one word: FAITHFUL or HALLUCINATED."
                ),
            },
            {"role": "user", "content": task},
        ]
        if hasattr(self.tokenizer, "apply_chat_template"):
            try:
                return self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                pass
        return (
            "<system>" + messages[0]["content"] + "</system>\n"
            + "<user>" + messages[1]["content"] + "</user>\n<assistant>"
        )

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        prompts = [self.make_prompt(row) for _, row in frame.iterrows()]
        output = np.full((len(frame), GEMMA_FEATURE_DIM), 0.5, dtype=np.float32)
        output[:, 1] = 0.0
        output[:, 2] = 0.0
        with torch.inference_mode():
            for start in range(0, len(prompts), self.batch_size):
                batch = prompts[start:start + self.batch_size]
                inputs = self.tokenizer(
                    batch, padding=True, truncation=True,
                    max_length=1536, return_tensors="pt"
                ).to("cuda")
                gen_kwargs = dict(
                    max_new_tokens=4,
                    return_dict_in_generate=True,
                    output_logits=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
                if self.num_samples > 1:
                    # Lower temperature (0.3 vs 0.7) so variance reduction
                    # affects the FIRST token only, not creative rewrites.
                    gen_kwargs.update(
                        do_sample=True, temperature=0.3, top_p=0.9,
                        num_return_sequences=self.num_samples,
                    )
                else:
                    gen_kwargs.update(do_sample=False)
                generated = self.model.generate(**inputs, **gen_kwargs)
                # logits[0] is the logits AFTER sampling/argmax of step 0.
                first_logits = generated.logits[0]
                probs = torch.softmax(first_logits.float(), dim=-1)
                answers = self.tokenizer.batch_decode(
                    generated.sequences[:, inputs["input_ids"].shape[1]:],
                    skip_special_tokens=True,
                )
                for offset in range(len(batch)):
                    soft_list, hard_list, margin_list = [], [], []
                    for sample_idx in range(self.num_samples):
                        idx = offset * self.num_samples + sample_idx
                        if idx >= probs.shape[0] or idx >= len(answers):
                            continue
                        p_faithful = float(probs[idx, self.faithful_token_ids].sum().clamp(min=0.0))
                        p_hall = float(probs[idx, self.hallucinated_token_ids].sum().clamp(min=0.0))
                        denom = p_faithful + p_hall
                        soft = p_hall / denom if denom > 1e-8 else 0.5
                        answer = answers[idx]
                        normalized = answer.upper()
                        if "HALLUCINATED" in normalized:
                            hard = 1.0
                            if soft < 0.5: soft = max(soft, 0.65)
                        elif "FAITHFUL" in normalized:
                            hard = 0.0
                            if soft > 0.5: soft = min(soft, 0.35)
                        else:
                            hard = 1.0 if soft >= 0.5 else 0.0
                        margin = abs(p_hall - p_faithful)
                        soft_list.append(soft)
                        hard_list.append(hard)
                        margin_list.append(margin)
                    if not soft_list:
                        output[start + offset] = [0.5, 0.0, 0.0]
                    else:
                        output[start + offset] = [
                            float(np.mean(soft_list)),
                            float(np.mean(hard_list)),
                            float(np.mean(margin_list)),
                        ]
        return output

    def close(self) -> None:
        del self.model
        if hasattr(self, "tokenizer"):
            del self.tokenizer
        empty_cache()


## Lexical & Combined Feature Engineering

Generates TF-IDF features and concatenates them with numeric lexical, reference memory, and model verifier features.


In [ ]:
def lexical_features(frame: pd.DataFrame) -> np.ndarray:
    rows = []
    for row in frame.itertuples():
        prompt_words = set(key_text(row.prompt_bn).split())
        answer_words = set(key_text(row.response_bn).split())
        context_words = set(key_text(row.context).split()) if row.has_context else set()
        answer_chars = len(row.response_bn)
        digit_ratio = sum(ch.isdigit() for ch in row.response_bn) / max(1, answer_chars)
        rows.append([
            float(row.has_context),
            float(len(row.prompt_bn)),
            float(len(row.response_bn)),
            float(len(row.context)),
            len(answer_words & context_words) / max(1, len(answer_words)),
            len(answer_words & prompt_words) / max(1, len(answer_words)),
            len(row.response_bn) / max(1, len(row.prompt_bn)),
            float(len(answer_words)),
            float(digit_ratio),
            float(bool(re.search(r"[কখগঘঙচছজঝঞটঠডঢণতথদধনপফবভমযরলশষসহড়ঢ়য়ৎ০-৯0-9A-Za-z]", row.response_bn))),
        ])
    return np.asarray(rows, dtype=np.float32)



def make_sparse_features(
    train: pd.DataFrame,
    frame: pd.DataFrame,
    vectorizer: TfidfVectorizer,
    scaler: StandardScaler,
    verifier_features: np.ndarray,
    exclude_same_index: bool = False,
) -> csr_matrix:
    sparse = vectorizer.transform(make_text(frame))
    numeric = np.hstack([
        lexical_features(frame),
        answer_reference_features(train, frame, exclude_same_index=exclude_same_index),
        verifier_features,
    ])
    return hstack([sparse, csr_matrix(scaler.transform(numeric))], format="csr")



## Model Training & Dual Thresholding

Functions to train the logistic regression classifier and select optimal dual decision thresholds (open-book vs. closed-book) to maximize class-0 F1.


In [ ]:
def choose_threshold(y_hallucinated: np.ndarray, probabilities: np.ndarray) -> float:
    """One-shot argmax over an OOF grid. Used by single-shot mode and by the
    nested aggregator (one call per seed). Returns 0.5 if the subset is empty
    or has no positive labels.
    """
    if len(y_hallucinated) == 0 or y_hallucinated.sum() == 0:
        return 0.5
    candidates = np.linspace(0.05, 0.95, 181)
    return float(
        max(candidates, key=lambda value: f1_score(y_hallucinated, probabilities >= value, zero_division=0))
    )


def choose_dual_thresholds_nested(
    y_hallucinated: np.ndarray,
    seed_oof_list: list[np.ndarray],
    has_context_mask: np.ndarray,
    final_oof: np.ndarray,
) -> tuple[float, float, float]:
    """Nested-CV threshold: pick open/closed thresholds per seed (each seed is
    itself a 5-fold CV), then median-aggregate across seeds for robustness.
    Score the final thresholds against the averaged OOF predictions so the
    reported F1 stays comparable to the single-shot path.
    """
    open_mask = has_context_mask.astype(bool)
    closed_mask = ~open_mask
    open_thrs: list[float] = []
    closed_thrs: list[float] = []
    for oof in seed_oof_list:
        open_thrs.append(choose_threshold(y_hallucinated[open_mask], oof[open_mask]) if open_mask.any() else 0.5)
        closed_thrs.append(choose_threshold(y_hallucinated[closed_mask], oof[closed_mask]) if closed_mask.any() else 0.5)
    thr_open = float(np.median(open_thrs))
    thr_closed = float(np.median(closed_thrs))
    preds = apply_dual_thresholds(final_oof, open_mask, thr_open, thr_closed)
    score = f1_score(y_hallucinated, preds, zero_division=0)
    return thr_open, thr_closed, float(score)


def choose_dual_thresholds(
    y_hallucinated: np.ndarray,
    probabilities: np.ndarray,
    has_context_mask: np.ndarray,
) -> tuple[float, float, float]:
    open_mask = has_context_mask.astype(bool)
    closed_mask = ~open_mask
    thr_open = choose_threshold(y_hallucinated[open_mask], probabilities[open_mask]) if open_mask.any() else 0.5
    thr_closed = (
        choose_threshold(y_hallucinated[closed_mask], probabilities[closed_mask]) if closed_mask.any() else 0.5
    )
    preds = apply_dual_thresholds(probabilities, open_mask, thr_open, thr_closed)
    score = f1_score(y_hallucinated, preds, zero_division=0)
    return thr_open, thr_closed, float(score)


def apply_dual_thresholds(
    probabilities: np.ndarray,
    has_context_mask: np.ndarray,
    thr_open: float,
    thr_closed: float,
) -> np.ndarray:
    preds = np.zeros(len(probabilities), dtype=int)
    open_mask = has_context_mask.astype(bool)
    preds[open_mask] = (probabilities[open_mask] >= thr_open).astype(int)
    preds[~open_mask] = (probabilities[~open_mask] >= thr_closed).astype(int)
    return preds



def fit_classifier(train_features, labels):
    class EnsembleClassifier:
        """Ensemble of Logistic Regression models with different regularization strengths."""
        def __init__(self, C_vals: list[float] | None = None):
            if C_vals is None:
                C_vals = [0.1, 1.0, 10.0]
            self.models = [
                LogisticRegression(
                    C=c, class_weight="balanced", max_iter=3000, solver="saga", n_jobs=-1, random_state=2026
                )
                for c in C_vals
            ]

        def fit(self, X, y):
            for model in self.models:
                model.fit(X, y)
            # Expose the coefficients of the medium model (C=1.0) for feature importance plotting.
            self.coef_ = self.models[1].coef_

        def predict_proba(self, X):
            probs = [model.predict_proba(X) for model in self.models]
            return np.mean(probs, axis=0)

    # Ensembling logistic regressions with C values: 0.1 * LR_C, LR_C, 10.0 * LR_C.
    model = EnsembleClassifier(C_vals=[0.1 * LR_C, LR_C, 10.0 * LR_C])
    model.fit(train_features, labels)
    return model



## Grouped Cross-Validation Loop

Performs StratifiedGroupKFold validation grouped by prompt to prevent prompt leakage across training and validation folds.


In [ ]:
def cross_validated_threshold(
    frame: pd.DataFrame, labels: np.ndarray, verifier_features: np.ndarray
) -> tuple[float, float, float, np.ndarray]:
    seeds = [2026, 2027, 2028, 2029, 2030]
    num_seeds = len(seeds)
    oof_accum = np.zeros(len(labels), dtype=float)
    groups = frame["prompt_key"].to_numpy()
    has_context_mask = frame["has_context"].to_numpy()
    seed_oof_list: list[np.ndarray] = []

    for seed in seeds:
        folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        oof_seed = np.zeros(len(labels), dtype=float)
        for train_index, valid_index in folds.split(frame, labels, groups):
            train_fold = frame.iloc[train_index]
            valid_fold = frame.iloc[valid_index]
            vectorizer = TfidfVectorizer(
                analyzer="char_wb", ngram_range=(2, 5), min_df=1, max_features=TFIDF_MAX_FEATURES, sublinear_tf=True
            ).fit(make_text(train_fold))
            train_numeric = np.hstack([
                lexical_features(train_fold),
                answer_reference_features(train_fold, train_fold, exclude_same_index=True),
                verifier_features[train_index],
            ])
            valid_numeric = np.hstack([
                lexical_features(valid_fold),
                answer_reference_features(train_fold, valid_fold),
                verifier_features[valid_index],
            ])
            scaler = StandardScaler().fit(train_numeric)
            train_features = hstack([
                vectorizer.transform(make_text(train_fold)), csr_matrix(scaler.transform(train_numeric))
            ], format="csr")
            valid_features = hstack([
                vectorizer.transform(make_text(valid_fold)), csr_matrix(scaler.transform(valid_numeric))
            ], format="csr")
            model = fit_classifier(train_features, labels[train_index])
            oof_seed[valid_index] = model.predict_proba(valid_features)[:, 1]
        oof_accum += oof_seed
        seed_oof_list.append(oof_seed)

    oof = oof_accum / num_seeds
    if THRESHOLD_METHOD == "nested":
        thr_open, thr_closed, cv_f1 = choose_dual_thresholds_nested(
            labels, seed_oof_list, has_context_mask, oof
        )
    else:
        thr_open, thr_closed, cv_f1 = choose_dual_thresholds(labels, oof, has_context_mask)
    # Also report subset diagnostics.
    open_mask = has_context_mask.astype(bool)
    if open_mask.any():
        open_f1 = f1_score(
            labels[open_mask],
            oof[open_mask] >= thr_open,
            zero_division=0,
        )
    else:
        open_f1 = float("nan")
    if (~open_mask).any():
        closed_f1 = f1_score(
            labels[~open_mask],
            oof[~open_mask] >= thr_closed,
            zero_division=0,
        )
    else:
        closed_f1 = float("nan")
    print(
        f"OOF F1 (method={THRESHOLD_METHOD}): {cv_f1:.4f}; "
        f"open-book F1: {open_f1:.4f} @ thr={thr_open:.3f}; "
        f"closed-book F1: {closed_f1:.4f} @ thr={thr_closed:.3f}"
    )
    return thr_open, thr_closed, cv_f1, oof



## Main Pipeline Runner

Runs the end-to-end training, validation, test predictions, and output generation.


In [ ]:
def run(args: argparse.Namespace) -> dict:
    train, test = load_data(args.train, args.test)
    labels = (train["label"].to_numpy() == 0).astype(int)
    print(
        f"Train={len(train)} Test={len(test)} "
        f"open_train={int(train['has_context'].sum())} closed_train={int((~train['has_context']).sum())} "
        f"open_test={int(test['has_context'].sum())} closed_test={int((~test['has_context']).sum())}"
    )

    verifier = Verifier(args.embed_model, args.nli_model, args.batch_size)
    print("Computing E5+NLI verifier features (train)...")
    train_verifier_features = verifier.transform(train)
    print("Computing E5+NLI verifier features (test)...")
    train_top_evidence = dict(getattr(verifier, "last_top_evidence", {}) or {})
    test_verifier_features = verifier.transform(test)
    verifier.close()

    if GEMMA_USE_RETRIEVED_EVIDENCE and (train_top_evidence or getattr(verifier, "last_top_evidence", None)):
        train_ev_map = train_top_evidence or {}
        test_ev_map = getattr(verifier, "last_top_evidence", {}) or {}
        train = train.copy()
        test = test.copy()
        train["retrieved_evidence"] = [
            train_ev_map.get(idx, "") for idx in train.index
        ]
        test["retrieved_evidence"] = [
            test_ev_map.get(idx, "") for idx in test.index
        ]
        n_ev_train = int((train["retrieved_evidence"].astype(bool)).sum())
        n_ev_test = int((test["retrieved_evidence"].astype(bool)).sum())
        msg = "Attaching retrieved evidence to " + str(n_ev_train) + "/" + str(len(train)) + " train rows and " + str(n_ev_test) + "/" + str(len(test)) + " test rows for Gemma."
        print(msg)

    judge_path = getattr(args, "judge_model", None) or getattr(args, "gemma_model", None) or JUDGE_MODEL_PATH
    judge_batch = getattr(args, "judge_batch_size", None) or getattr(args, "gemma_batch_size", None) or JUDGE_BATCH_SIZE
    if judge_path:
        label = "causal-LM judge (" + JUDGE_KIND + ")"
        print("Running " + label + " on " + str(len(train) + len(test)) + " rows; model=" + str(Path(judge_path).name))
        gemma_judge = CausalLMJudge(judge_path, batch_size=judge_batch, kind=JUDGE_KIND)
        train_gemma = gemma_judge.transform(train)
        test_gemma = gemma_judge.transform(test)
        gemma_judge.close()
        train_verifier_features = np.hstack([train_verifier_features, train_gemma])
        test_verifier_features = np.hstack([test_verifier_features, test_gemma])

    vectorizer = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), min_df=1, max_features=TFIDF_MAX_FEATURES, sublinear_tf=True
    ).fit(make_text(train))
    train_numeric = np.hstack([
        lexical_features(train),
        answer_reference_features(train, train, exclude_same_index=True),
        train_verifier_features,
    ])
    scaler = StandardScaler().fit(train_numeric)
    thr_open, thr_closed, cv_f1, oof = cross_validated_threshold(train, labels, train_verifier_features)
    thr_open = float(np.clip(thr_open + THR_OPEN_DELTA, 0.01, 0.99))
    thr_closed = float(np.clip(thr_closed + THR_CLOSED_DELTA, 0.01, 0.99))
    train_features = make_sparse_features(
        train, train, vectorizer, scaler, train_verifier_features, exclude_same_index=True
    )
    print(
        f"OOF F1 for hallucinated class: {cv_f1:.4f}; "
        f"threshold_open={thr_open:.3f}; threshold_closed={thr_closed:.3f}"
    )

    test_features = make_sparse_features(train, test, vectorizer, scaler, test_verifier_features)
    model = fit_classifier(train_features, labels)
    hallucination_probability = model.predict_proba(test_features)[:, 1]
    predictions = apply_dual_thresholds(
        hallucination_probability, test["has_context"].to_numpy(), thr_open, thr_closed
    )

    # Deterministic reuse of exact labeled answer matches for repeated prompts.
    known = answer_reference_features(train, test)
    exact_faithful_mask = known[:, 3] == 1
    exact_hallucinated_mask = known[:, 4] == 1
    exact_faithful_count = int(exact_faithful_mask.sum())
    exact_hallucinated_count = int(exact_hallucinated_mask.sum())
    exact_total = exact_faithful_count + exact_hallucinated_count
    print(
        "Exact-match override fires on " + str(exact_total) + "/" + str(len(test)) + " test rows "
        "(" + format(exact_total / max(len(test), 1), ".1%") + "); "
        "faithful=" + str(exact_faithful_count) + ", hallucinated=" + str(exact_hallucinated_count)
    )
    predictions[exact_faithful_mask] = 0
    predictions[exact_hallucinated_mask] = 1

    if NEAR_MATCH_OVERRIDE:
        best_sim = known[:, 5]
        best_label_is_hallucinated = known[:, 6] == 1
        candidate_mask = (~exact_faithful_mask) & (~exact_hallucinated_mask) & (best_sim >= NEAR_MATCH_SIM_THRESHOLD)
        model_prob = hallucination_probability
        confident_match = (best_label_is_hallucinated & (model_prob >= 0.5 + NEAR_MATCH_LABEL_CONFIDENCE)) | ((~best_label_is_hallucinated) & (model_prob <= 0.5 - NEAR_MATCH_LABEL_CONFIDENCE))
        fire_mask = candidate_mask & (~confident_match)
        n_fuzzy = int(fire_mask.sum())
        if n_fuzzy:
            predictions[fire_mask & best_label_is_hallucinated] = 1
            predictions[fire_mask & (~best_label_is_hallucinated)] = 0
        print(
            "Near-match override fires on " + str(n_fuzzy) + "/" + str(len(test)) + " test rows "
        )

    submission = pd.DataFrame({"id": test["id"], "label": 1 - predictions})
    submission.to_csv(args.output, index=False)
    print(f"Wrote {args.output} with {len(submission)} rows.")
    print(
        "Predicted label balance:",
        submission["label"].value_counts(normalize=True).sort_index().to_dict(),
    )
    return {
        "train": train,
        "test": test,
        "labels": labels,
        "oof": oof,
        "thr_open": thr_open,
        "thr_closed": thr_closed,
        "cv_f1": cv_f1,
        "model": model,
        "vectorizer": vectorizer,
        "scaler": scaler,
        "train_verifier_features": train_verifier_features,
        "test_verifier_features": test_verifier_features,
        "predictions": predictions,
        "test_probs": hallucination_probability,
    }



## Pipeline configuration

Reuses paths set above.


In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    train=TRAIN_PATH,
    test=TEST_PATH,
    output="submission.csv",
    embed_model=EMBED_MODEL_PATH,
    nli_model=NLI_MODEL_PATH,
    batch_size=BATCH_SIZE,

    gemma_model=GEMMA_MODEL_PATH,
    gemma_batch_size=GEMMA_BATCH_SIZE,

)
print(args)


## Train, validate, and create submission

Run the pipeline. OOF F1 score is printed for class-0 (hallucinated). Returns evaluation variables for subsequent plotting.


In [ ]:
results = run(args)

## OOF Predictions & Thresholds Diagnostics

KDE distribution of Out-Of-Fold predicted probabilities colored by their true label (0 = Hallucinated, 1 = Faithful). The dashed red lines show the selected dual thresholds.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract outputs
labels_arr = results["labels"]
oof_arr = results["oof"]
thr_open = results["thr_open"]
thr_closed = results["thr_closed"]
has_context_mask = results["train"]["has_context"].to_numpy().astype(bool)

# Set style
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot open-book (with context) OOF probabilities
open_probs_0 = oof_arr[has_context_mask & (labels_arr == 0)]
open_probs_1 = oof_arr[has_context_mask & (labels_arr == 1)]

if len(open_probs_0) > 0 or len(open_probs_1) > 0:
    sns.kdeplot(open_probs_0, fill=True, label="True Hallucinated (label=0)", color="coral", ax=axes[0], warn_singular=False)
    sns.kdeplot(open_probs_1, fill=True, label="True Faithful (label=1)", color="skyblue", ax=axes[0], warn_singular=False)
    axes[0].axvline(thr_open, color="red", linestyle="--", linewidth=2, label=f"Threshold: {thr_open:.3f}")
    axes[0].set_title(f"Open-Book (With Context) OOF Predictions\n(F1 Target: Class-0)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Predicted P(Hallucinated)")
    axes[0].set_ylabel("Density")
    axes[0].legend(loc="upper right")

# Plot closed-book (without context) OOF probabilities
closed_probs_0 = oof_arr[~has_context_mask & (labels_arr == 0)]
closed_probs_1 = oof_arr[~has_context_mask & (labels_arr == 1)]

if len(closed_probs_0) > 0 or len(closed_probs_1) > 0:
    sns.kdeplot(closed_probs_0, fill=True, label="True Hallucinated (label=0)", color="coral", ax=axes[1], warn_singular=False)
    sns.kdeplot(closed_probs_1, fill=True, label="True Faithful (label=1)", color="skyblue", ax=axes[1], warn_singular=False)
    axes[1].axvline(thr_closed, color="red", linestyle="--", linewidth=2, label=f"Threshold: {thr_closed:.3f}")
    axes[1].set_title(f"Closed-Book (No Context) OOF Predictions\n(F1 Target: Class-0)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Predicted P(Hallucinated)")
    axes[1].set_ylabel("Density")
    axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig("oof_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()


## Feature Importance (Numeric Coefficients)

Horizontal bar plot of the logistic regression coefficients for the numeric features. A positive coefficient indicates association with the hallucinated class (class-0).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

num_verifier = NLI_FEATURE_DIM
num_gemma = results["train_verifier_features"].shape[1] - num_verifier

feature_names = [
    "has_context", "len(prompt)", "len(response)", "len(context)",
    "token_overlap_context", "token_overlap_prompt", "len_ratio_resp_prompt",
    "num_unique_resp_words", "digit_ratio", "has_bengali_alphanumeric",
    "has_ref_prompts", "max_sim_to_ref_faithful", "max_sim_to_ref_hallucinated",
    "exact_faithful_ref_match", "exact_hallucinated_ref_match", "best_ref_sim",
    "best_ref_label_is_hallucinated",
    "max_entailment", "mean_entailment", "min_entailment",
    "max_contradiction", "mean_contradiction", "entail_contrad_margin",
    "rate_contradiction_gt_0.5", "rate_entailment_gt_0.5", "max_evidence_similarity",
    "mean_evidence_similarity", "query_answer_similarity", "answer_evidence_overlap"
]
if PER_CLAIM_NLI:
    feature_names.extend([
        "min_claim_entailment", "mean_claim_entailment",
        "max_claim_contradiction", "claim_contradiction_rate"
    ])
if num_gemma > 0:
    feature_names.extend([
        "gemma_soft_p_hallucinated", "gemma_hard_vote", "gemma_logit_margin"
    ])

coefs = results["model"].coef_[0][-len(feature_names):]
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs,
    "Absolute_Coefficient": np.abs(coefs)
}).sort_values("Absolute_Coefficient", ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(
    x="Coefficient", y="Feature", data=importance_df, 
    palette=sns.diverging_palette(10, 240, n=len(feature_names))
)
plt.title("Meta-Classifier Numeric Feature Coefficients\n(Positive = Associated with Hallucination)", fontsize=12, fontweight="bold")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


## Verify submission

Validate the submission file format, length, columns, and value bounds.


In [ ]:
import pandas as pd
from pathlib import Path

submission = pd.read_csv("submission.csv")
assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(pd.read_csv(TEST_PATH))
assert set(submission["label"].unique()).issubset({0, 1})
assert submission["label"].dtype.kind in "iu"
print(submission.head())
print("Rows:", len(submission))
print("Label balance:", submission["label"].value_counts(normalize=True).sort_index().to_dict())
print("Ready to submit:", Path("submission.csv").resolve())
